<a href="https://colab.research.google.com/github/chenwh0/AudioSentimentAnalysisEnd2EndPipeline/blob/main/sample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importing Libraries

In [ ]:
!pip install tqdm -q

In [ ]:
from datasets import load_dataset
from transformers import pipeline
import torch
import pandas
from tqdm.auto import tqdm
from sklearn.metrics import classification_report

# Check if Cuda is available

In [ ]:
device = 0 if torch.cuda.is_available() else -1

# Load the Dataset

In [ ]:
try:
    dataset = load_dataset("DynamicSuperb/Sentiment_Analysis_SLUE-VoxCeleb", split="test")
    print(dataset)
    original_dataframe = pandas.DataFrame(dataset)
    display(original_dataframe)
except Exception as e:
    print(f"Failed to load dataset: {e}")
    exit()

README.md:   0%|          | 0.00/392 [00:00<?, ?B/s]

data/test-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  270MB            

data/test-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/test-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  251MB            

data/test-00001-of-00002.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/3553 [00:00<?, ? examples/s]

Dataset({
    features: ['audio', 'file', 'instruction', 'label'],
    num_rows: 3553
})


,audio,file,instruction,label
0,<datasets.features._torchcodec.AudioDecoder ob...,id10270_5r0dWxy17C8_00001,Determine the sentiment of the speech as eithe...,Neutral
1,<datasets.features._torchcodec.AudioDecoder ob...,id10270_5r0dWxy17C8_00002,Determine the sentiment of the speech as eithe...,Neutral
2,<datasets.features._torchcodec.AudioDecoder ob...,id10270_5r0dWxy17C8_00004,Determine the sentiment of the speech as eithe...,Neutral
3,<datasets.features._torchcodec.AudioDecoder ob...,id10270_5r0dWxy17C8_00005,Determine the sentiment of the speech as eithe...,Neutral
4,<datasets.features._torchcodec.AudioDecoder ob...,id10270_5r0dWxy17C8_00006,Can you identify the sentiment of the speech a...,Neutral
...,...,...,...,...
3548,<datasets.features._torchcodec.AudioDecoder ob...,id10309_vobW27_-JyQ_00011,Determine the sentiment of the speech as eithe...,Negative
3549,<datasets.features._torchcodec.AudioDecoder ob...,id10309_vobW27_-JyQ_00012,Analyze the sentiment of the speech and classi...,Neutral
3550,<datasets.features._torchcodec.AudioDecoder ob...,id10309_vobW27_-JyQ_00013,Determine the sentiment of the speech as eithe...,Negative
3551,<datasets.features._torchcodec.AudioDecoder ob...,id10309_vobW27_-JyQ_00014,Analyze the sentiment of the speech and classi...,Neutral


In [ ]:
# All possible label classes
display(original_dataframe["label"].unique())

array(['Neutral', 'Negative', 'Disagreement', 'Positive', '<mixed>'],
      dtype=object)

# Load the Models

In [ ]:
asr_models = [
    "AventIQ-AI/whisper-audio-to-text",
    "facebook/s2t-small-librispeech-asr"
]

In [ ]:
sentiment_analysis_models = [
    "siebert/sentiment-roberta-large-english",
    "cardiffnlp/twitter-roberta-base-sentiment-latest",
    "tabularisai/multilingual-sentiment-analysis"
]

In [ ]:
asr = pipeline(
    "automatic-speech-recognition",
    model=asr_models[0],
    device=device
)

config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  145MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.80k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

In [ ]:
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model=sentiment_analysis_models[0],
    device=device
)

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.42GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

tokenizer_config.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

# Inference

In [ ]:
class DatasetLoader:
    def __init__(self, dataset_name="DynamicSuperb/Sentiment_Analysis_SLUE-VoxCeleb", split="test"):
        self.dataset_name = dataset_name
        self.split = split

    def load(self) -> pandas.DataFrame:
        try:
            dataset = load_dataset(self.dataset_name, split=self.split)
            print(dataset)
            return pandas.DataFrame(dataset)
        except Exception as error:
            raise RuntimeError(f"Failed to load dataset '{self.dataset_name}': {error}") from error

In [ ]:
class AudioSentimentAnalyzer:
    def __init__(self, asr_model_name: str, sentiment_model_name: str):
        self.asr_model_name = asr_model_name
        self.sentiment_model_name = sentiment_model_name

        self.device = self._get_device()
        self.speech_recognizer = self._build_speech_recognizer()
        self.sentiment_analyzer = self._build_sentiment_analyzer()

    @staticmethod
    def _get_device() -> int:
        return 0 if torch.cuda.is_available() else -1
    def _build_speech_recognizer(self):
        return pipeline(
            "automatic-speech-recognition",
            model=self.asr_model_name,
            device=self.device
        )
    def _build_sentiment_analyzer(self):
        return pipeline(
            "sentiment-analysis",
            model=self.sentiment_model_name,
            device=self.device
        )

    def transcribe(self, audio):
        return self.speech_recognizer(audio)["text"]

    def predict_sentiment(self, transcription: str):
        result = self.sentiment_analyzer(transcription)[0]
        return result["label"], result["score"]

    def analyze(self, dataframe: pandas.DataFrame) -> pandas.DataFrame:

        results = pandas.DataFrame({"audio": dataframe["audio"], "actual_sentiment": dataframe["label"]})

        tqdm.pandas()
        results["transcription"] = results["audio"].progress_apply(self.transcribe)
        predictions = results["transcription"].progress_apply(self.predict_sentiment)
        results[["predicted_sentiment", "sentiment_score"]] = pandas.DataFrame(predictions.tolist(), index=results.index)
        return results

In [ ]:
ASR_MODELS = [
    "AventIQ-AI/whisper-audio-to-text",
    "facebook/s2t-small-librispeech-asr"
]

SENTIMENT_MODELS = [
    "siebert/sentiment-roberta-large-english",
    "cardiffnlp/twitter-roberta-base-sentiment-latest",
    "tabularisai/multilingual-sentiment-analysis"
]

def run_analyzers():
    dataset_loader = DatasetLoader()
    original_dataframe = dataset_loader.load()
    models_results = {}
    for asr_model in ASR_MODELS:
        for sentiment_model in SENTIMENT_MODELS:
            model_names = f"ASR: {asr_model}. Sentiment: {sentiment_model}"
            print(model_names)
            analyzer = AudioSentimentAnalyzer(asr_model, sentiment_model)
            results = analyzer.analyze(original_dataframe)
            models_results[model_names] = results
            display(results)
    return models_results

if __name__ == "__main__":
    models_results = run_analyzers()

Dataset({
    features: ['audio', 'file', 'instruction', 'label'],
    num_rows: 3553
})
ASR: AventIQ-AI/whisper-audio-to-text. Sentiment: siebert/sentiment-roberta-large-english


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

  0%|          | 0/3553 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [ ]:
# Get accuracy scores
for model_name, dataframe in models_results.items():
    print(f"\n{model_name}")
    print(classification_report(dataframe["actual_sentiment"].lower(), dataframe["predicted_sentiment"].lower()))
    print()